In [ ]:
from langchain_ollama import ChatOllama
############################################################################################################################################################
# DEMOCRATIC WORKFLOW
############################################################################################################################################################
from sklearn.metrics import classification_report
from tools import ConfusionMatrix
from main import SummaryTMP
import pandas as pd
import json
import matplotlib.pyplot as plt

# State rows to keep
keep=['accession', 'thermal_range', 'thermal_source', 'thermal_reasoning', 'thermal_confidence', 'thermal_source', 'inference_type', 'thermal_paper', 'host','taxonomic_level','host_reasoning','host_confidence','host_source','host_paper','duration', 'nodes', 'timings']

#Getting accessions
ex_df=pd.read_csv('data/example/expanded_example_data.csv')
accessions=ex_df['Accession']


all_df = pd.DataFrame()

# Classifying each accession
preds=[]
for acc in accessions:
    print(f'Accession: {acc}')
    result = SummaryTMP(acc, "gemma4:e4b")


    preds.append(result.get('thermal_range', None))
    print(f'\n{acc}: {result.get("thermal_range", None)}')
    df = pd.DataFrame([result])

    # serialize complex fields
    df["metadata"] = df["metadata"].apply(json.dumps)
    df["timings"] = df["timings"].apply(json.dumps)
    df["nodes"] = df["nodes"].apply(json.dumps)

    all_df = pd.concat([all_df, df], ignore_index=True)



all_df["thermal_range"] = preds

all_df.to_csv('results/summary_result.csv', index=False)

# Create confusion matrix
actual=ex_df['Thermal Range']
pred=all_df['thermal_range']

#Clean out None
actual_clean = actual.fillna("unknown")
pred_clean = pred.fillna("unknown")

plot=ConfusionMatrix(actual_clean, pred_clean)
plot.plot(cmap='Blues')
plt.savefig('results/summary_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


# Total duration
import pandas as pd
df=pd.read_csv('results/summary_result.csv')
total_duration=0
for i in df['duration']:
    total_duration+=i
print(f'Total Duration: {round((total_duration/60),2)} minutes')

# Get the classification report
classification_report=classification_report(actual_clean, pred_clean)

with open('results/summary_classification_report.txt', 'w') as f:
    f.write(classification_report)
    f.write(f'\n\n Total Duration: {round((total_duration/60),2)} minutes')


In [2]:
from src.graphs import VisualiseGraph, SummaryGraph, DemocraticGraph, FastGraph
VisualiseGraph(FastGraph)
VisualiseGraph(DemocraticGraph)
VisualiseGraph(SummaryGraph)


In [1]:
from main import FastTMP
fast_result=FastTMP('A7XXB7', 'gemma4:e4b')
fast_result

/opt/homebrew/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ClassifyThermalMetadata: thermal_range='thermophile' temperature=None inference_type='inferred' thermal_reasoning="The organism listed is 'Thermus phage P23-45' and the host is 'Thermus thermophilus'. The genus 'Thermus' is associated with thermophilic organisms, implying a thermophilic thermal range." thermal_confidence='medium' thermal_found=True




{'model': 'gemma4:e4b',
 'accession': 'A7XXB7',
 'phage': 'Thermus phage P23-45',
 'metadata': {'uid': '1679373715',
  'accession': 'A7XXB7',
  'title': 'RecName: Full=Terminase, large subunit; AltName: Full=DNA-packaging protein; AltName: Full=Gene product 85; Short=gp85; Includes: RecName: Full=ATPase; Includes: RecName: Full=Endonuclease',
  'organism': 'Thermus phage P23-45',
  'taxonId': 2914006,
  'sequence_length': 485,
  'create_date': '2019/05/08',
  'update_date': '2026/01/28',
  'dbsource': 'swiss_prot',
  'extra': 'gi|1679373715|sp|A7XXB7.1|TERL_BP234',
  'host': 'Thermus thermophilus'},
 'host': None,
 'taxonomic_level': None,
 'host_reasoning': None,
 'host_confidence': None,
 'host_source': None,
 'host_found': False,
 'thermal_range': 'thermophile',
 'temperature': None,
 'thermal_reasoning': "The organism listed is 'Thermus phage P23-45' and the host is 'Thermus thermophilus'. The genus 'Thermus' is associated with thermophilic organisms, implying a thermophilic therma

In [1]:
from main import DemocraticTMP

democratic_result=DemocraticTMP('A7XXB7', 'gemma4:e4b')
democratic_result

/opt/homebrew/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


download: s3://pmc-oa-opendata/PMC6397560.1/PMC6397560.1.txt to data/accessions/A7XXB7/library/accession_lit/PMC6397560.1.txt
ClassifyHostMetadata: host='Thermus thermophilus' taxonomic_level='species' host_reasoning="The metadata explicitly contains the key 'host' with the value 'Thermus thermophilus'." host_found=True


download: s3://pmc-oa-opendata/PMC13150438.1/PMC13150438.1.txt to data/accessions/A7XXB7/library/host_lit/PMC13150438.1.txt
download: s3://pmc-oa-opendata/PMC13039148.1/PMC13039148.1.txt to data/accessions/A7XXB7/library/host_lit/PMC13039148.1.txt
ClassifyThermalMetadataVote: thermal_range='thermophile' temperature=None inference_type='inferred' thermal_reasoning="The organism listed is 'Thermus phage P23-45' and the host is 'Thermus thermophilus'. The genus 'Thermus' is associated with thermophilic organisms, implying a thermophilic thermal range." thermal_confidence='medium' thermal_found=True


Classification'thermophile'
ClassifyThermalLiteratureVotes: thermal_ran

{'model': 'gemma4:e4b',
 'accession': 'A7XXB7',
 'phage': 'Thermus phage P23-45',
 'metadata': {'uid': '1679373715',
  'accession': 'A7XXB7',
  'title': 'RecName: Full=Terminase, large subunit; AltName: Full=DNA-packaging protein; AltName: Full=Gene product 85; Short=gp85; Includes: RecName: Full=ATPase; Includes: RecName: Full=Endonuclease',
  'organism': 'Thermus phage P23-45',
  'taxonId': 2914006,
  'sequence_length': 485,
  'create_date': '2019/05/08',
  'update_date': '2026/01/28',
  'dbsource': 'swiss_prot',
  'extra': 'gi|1679373715|sp|A7XXB7.1|TERL_BP234',
  'host': 'Thermus thermophilus'},
 'host': 'Thermus thermophilus',
 'taxonomic_level': 'species',
 'host_reasoning': "The metadata explicitly contains the key 'host' with the value 'Thermus thermophilus'.",
 'host_confidence': None,
 'host_source': 'metadata',
 'host_found': True,
 'thermal_range': 'thermophile',
 'temperature': None,
 'thermal_reasoning': None,
 'thermal_confidence': None,
 'thermal_found': False,
 'paper_

In [1]:
# Test examples
from main import SummaryTMP

summary_result=SummaryTMP('A7XXB7', 'gemma4:e4b')
summary_result

/opt/homebrew/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


download: s3://pmc-oa-opendata/PMC6397560.1/PMC6397560.1.txt to data/accessions/A7XXB7/library/accession_lit/PMC6397560.1.txt
ClassifyHostMetadata: host='Thermus thermophilus' taxonomic_level='species' host_reasoning="The metadata explicitly contains the key 'host' with the value 'Thermus thermophilus'." host_found=True


download: s3://pmc-oa-opendata/PMC13150438.1/PMC13150438.1.txt to data/accessions/A7XXB7/library/host_lit/PMC13150438.1.txt
download: s3://pmc-oa-opendata/PMC13039148.1/PMC13039148.1.txt to data/accessions/A7XXB7/library/host_lit/PMC13039148.1.txt


{'model': 'gemma4:e4b',
 'accession': 'A7XXB7',
 'phage': 'Thermus phage P23-45',
 'metadata': {'uid': '1679373715',
  'accession': 'A7XXB7',
  'title': 'RecName: Full=Terminase, large subunit; AltName: Full=DNA-packaging protein; AltName: Full=Gene product 85; Short=gp85; Includes: RecName: Full=ATPase; Includes: RecName: Full=Endonuclease',
  'organism': 'Thermus phage P23-45',
  'taxonId': 2914006,
  'sequence_length': 485,
  'create_date': '2019/05/08',
  'update_date': '2026/01/28',
  'dbsource': 'swiss_prot',
  'extra': 'gi|1679373715|sp|A7XXB7.1|TERL_BP234',
  'host': 'Thermus thermophilus'},
 'host': 'Thermus thermophilus',
 'taxonomic_level': 'species',
 'host_reasoning': "The metadata explicitly contains the key 'host' with the value 'Thermus thermophilus'.",
 'host_confidence': None,
 'host_source': 'metadata',
 'host_found': True,
 'thermal_range': 'thermophile',
 'temperature': None,
 'thermal_reasoning': "The majority of the papers (PMC6397560.1.txt, 42104995.txt, PMC1314